# Дубликаты, пропуски и неинформативные данные

Когда мы начинаем исследовать датасет, мы должны «почистить» его от плохих значений, наличие которых может влиять на результат обучения модели.
Но в первую очередь мы должны убедиться, что в выборке отсутствуют повторяющиеся объекты.

In [ ]:
import pandas as pd

Наша задача — построить модель классификации, определяющую категорию цены подержанного автомобиля в зависимости от характеристик транспортного средства. 
Посмотрим, как проверить дублирующиеся строки на нашем датасете. 

Загрузим датасет.

In [ ]:
df = pd.read_csv('./data/vehicles_dataset.csv')
df.head()

Посмотрим на размер выборки. 


In [ ]:
print(f'Размер выбоpки: {df.shape[0]}, {df.shape[1]}')

В датасете 10 050 строк. Узнать, есть ли среди всех значений дублирующиеся, можно с помощью метода `duplicated`, который возвращает булевую маску: если для объекта условие выполняется, то для него вернётся `True`, если нет —`False`. 

In [ ]:
df.duplicated()

С помощью такой булевой маски мы можем отфильтровать необходимые значения. Отфильтруем только те строки, которые помечены как дублирующиеся. 

In [ ]:
df[df.duplicated()]

Отобразилось 50 строк. Проверим одну из них. Например, отфильтруем по первому идентификатору в списке. 

In [ ]:
df[df['id'] == 7312641615]

Если попарно сравнить колонки, то увидим, что две строки идентичны.
Удалим лишние строки и сразу проверим размер датасета. 

In [ ]:
df = df.drop_duplicates()
df.shape

Видим, что осталось только 10 000 записей.


Теперь можно перейти к поиску пропущенных значений. Выведем информацию о датасете с помощью `info`. 

In [ ]:
print('Информация о выборке:\n')
df.info()

В колонке `non-null count` указано, для скольки объектов в каждом признаке заполнены значения.

`id` заполнены во всех объектах, `region` тоже. Но также много признаков, в которых есть незаполненные значения. Например, для количества цилиндров `cylinders` заполнено только 5920 значений. 

Чтобы оценить количество заполненных значений визуально, можно воспользоваться пакетом `missingno`. 

In [ ]:
#!pip install missingno
import missingno as msno  # https://github.com/ResidentMario/missingno

С помощью `msno.bar` получаем изображение, на котором можем видеть наиболее проседающие по заполненным значениям признаки. В глаза сразу бросается бесполезный признак `county`, который совсем не содержит информации.



In [ ]:
msno.bar(df);

С помощью `msno.matrix` можно вывести матрицу пропусков, которая повторяет структуру нашего датасета, где чёрным закрашены заполненные значения, а белым — пропуски.



In [ ]:
msno.matrix(df);

Существует два подхода для работы с пропущенными значениями:
- полное удаление строки или столбца, которые содержат пропуски;
- заполнение пропусков статистиками, например средним значением по признаку (если мы хотим оставить строку с полезной информацией).

Из матрицы пропусков мы видим, что пропуски разбросаны по всему датасету, поэтому мы не можем просто взять и удалить все строки, где они есть. Но при этом можем удалить неинформативные признаки, например `county` или `size`. 

Обычно неинформативным считается признак, в котором пропущено 20–30% значений. Пропуски — необязательно NaN: иногда это могут быть знаки вопроса, другие символы, которые означают, что в ячейке датафрейма располагается информация, которая будет бесполезна. В этом случае нужно внимательно изучить, что означают странные значения в датафрейме, и оценить их связь со значениями в других признаках. Иногда требуется посмотреть на исходные данные, если у вас есть доступ к источнику данных, например к базе данных компании. 

Посмотрим именно на те значения, которые реально пропущены. 
Перед тем как удалять признаки, выведем процент пропущенных значений в каждом признаке, чтобы понять, какие признаки можно удалить без жалости, а какие, возможно, стоит оставить. 

Для этого с помощью `isna` создадим маску, где у каждого объекта будет `True` или `False`, а затем просуммируем получившиеся значения (при работе с булевыми значениями `sum` посчитает количество входящих `True`). После этого разделим на всё количество объектов, домножим на 100 для перевода в проценты и отсортируем по возрастанию.


In [ ]:
missing_values = ((df.isna().sum() / len(df)) * 100).sort_values()

print('Процент пропущенных значений:')
missing_values

Проверим заполненность `county` ещё одним способом: используя `value_counts`, передадим параметр `dropna=False`. Параметр указывает, что незаполненные значения также нужно посчитать.




In [ ]:
df['county'].value_counts(dropna=False)

Действительно 10 000 NaN. Удалим эту колонку.

Делать это будем в новом датафрейме `df_clean`, чтобы сохранить исходный датасет. Так, если мы поймём, что преобразования, которые мы делали, нам не подходят, то сможем вернуться к исходному датасету.

In [ ]:
df_clean = df.copy()

Удалять столбцы можно с помощью метода `drop`, в который передаётся список столбцов на удаление. 

In [ ]:
df_clean = df_clean.drop(columns=['county'])
df_clean.head()

Оставшиеся столбцы содержат от 0 до 71% заполненных значений. Проверим, сколько полностью заполненных объектов во всём наборе данных. Напишем функцию на основе `dropna` — этот метод используется для удаления строк и столбцов с пустыми значениями.




In [ ]:
def print_useful_rows_info(df):
    print('Количество полностью заполненных объектов из всей выборки:', 
          len(df.dropna()))
    print('Процент полностью заполненных объектов из всей выборки:', 
          round(len(df.dropna()) / len(df) * 100, 2))
    

print_useful_rows_info(df_clean)

Полностью заполнено чуть больше 8% объектов, а значит, как мы и предполагали ранее, удалить такие объекты полностью мы не можем, иначе выборка уменьшится в 12 раз.

Следующим шагом удалим ещё семь столбцов, в которых более 20% пропущенных значений.

Сейчас мы выбрали порог в 20%, в других задачах он может отличаться в большую и даже в меньшую сторону. К примеру, несмотря на большое количество пропущенных значений, цвет автомобиля может сильно влиять на его стоимость, так как существуют премиальные цвета.


In [ ]:
df_clean = df_clean.drop(
    columns=['VIN', 'condition', 'cylinders', 'size', 'drive', 'paint_color', 'type']
)
print_useful_rows_info(df_clean)


После удаления количество полностью заполненных значений выросло почти до 92%. С таким процентом мы могли бы уже с помощью `dropna` удалить строки, в которых есть хоть один пропуск, и при этом потеряли бы только одну десятую часть выборки. Но по отдельности в каждом признаке осталось не так много пропусков, поэтому эти признаки пока оставим и будем придерживаться стратегии заполнения пустых значений.

В зависимости от того, к какому типу относится переменная, представляющая наш признак, существует несколько вариантов заполнения пропусков:
- Если это качественная, то есть категориальная переменная, то мы можем заполнить пропуски либо самым популярным значением в выборке, либо новым значением, которое ранее не встречалось в выборке. Или даже заполнить значениями, рассчитанными на основе других колонок, в которых может содержаться похожая информация.
- Если мы работаем с числами, то на помощь приходят меры центральной тенденции: среднее значение, мода или медиана.

Рассмотрим заполнение на примере нескольких колонок. Выведем количество значений в колонке `manufacturer` (производитель транспортного средства), учитывая и незаполненные значения. 

In [ ]:
df_clean['manufacturer'].value_counts(dropna=False)

Видим, что для 382 объектов производитель не заполнен. Но у нас есть колонки, из которых потенциально можно восстановить производителя — модель автомобиля и его описание. Посмотрим на значение этих двух признаков для тех объектов, в которых не заполнена модель. 


In [ ]:
df_clean[df_clean['manufacturer'].isna()][['model', 'description']]

По той малой части, что отобразилась, можно подтвердить, что производитель транспортного средства часто указан в наименовании модели или в описании. Например, в строке с индексом 9985 легко узнать Suzuki. 

Но чтобы сделать такое заполнение пропущенных значений, нужно верно обработать строки и проверить производителей по имеющемуся словарю — в общем, потрудиться чуть больше, чем мы предполагаем. Поэтому сейчас просто заполним пропущенные значения  новым значением other, или «другой». Сделаем это с помощью `fillna`:




In [ ]:
df_clean['manufacturer'] = df_clean['manufacturer'].fillna('other')

Проверим, используя `isna`, что пропущенных значений в колонке с производителем не осталось.

In [ ]:
df_clean['manufacturer'].isna().sum()

Так и есть, всё заполнено.

Теперь посмотрим на числовые переменные. Выведем количество значений  в колонке с количеством пройденных миль, включая пустые. 

In [ ]:
df_clean['odometer'].value_counts(dropna=False)

43 объекта не содержат информацию о пробеге. Посмотрим на эти строки внимательнее.




In [ ]:
df_clean[df_clean['odometer'].isna()]

Числовые пропуски  обычно заполняются мерами центральной тенденции. В зависимости от типа признака и его распределения можно выбрать наиболее подходящую меру. 

Заполним все пропуски в пробеге средним значением. Сначала посмотрим на него.

In [ ]:
int(df_clean['odometer'].mean())

Через `fillna` передадим среднее значение как параметр. 

In [ ]:
df_clean['odometer'] = df_clean['odometer'].fillna(int(df_clean['odometer'].mean()))

Проверим, что незаполненных значений не осталось.





In [ ]:
df_clean['odometer'].isna().sum()

В датафрейме ещё остались пропуски, но пока остановимся на двух примерах и пойдём исследовать и преобразовывать данные дальше. Но сначала запишем изменения в новый файл `df_clean.csv`.

In [ ]:
df_clean.to_csv('./data/df_clean.csv', index=False)